# OOD Evaluation: NAICS-2 Cross-Encoder

In [1]:
!pip install -q transformers sentencepiece scikit-learn pandas tqdm

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print("Drive mounted at /content/drive")
except Exception as e:
    DRIVE_MOUNTED = False
    print(f"Drive not mounted ({e}). Falling back to local paths.")


Mounted at /content/drive
Drive mounted at /content/drive


In [3]:
import os
import re
import gc
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Using device: cuda
GPU: Tesla T4
Total VRAM: 15.64 GB


In [4]:
MODEL_NAME      = "microsoft/deberta-v3-large"

DRIVE_MODEL_PATH = "/content/drive/MyDrive/naics2_large_run3_checkpoints_seed42/best_model.pt"
LOCAL_MODEL_PATH = "best_model.pt"
if DRIVE_MOUNTED and os.path.exists(DRIVE_MODEL_PATH):
    MODEL_PATH = DRIVE_MODEL_PATH
else:
    MODEL_PATH = LOCAL_MODEL_PATH

OOD_PATH        = "ood_dataset_official.csv"
TAXONOMY_PATH   = "ExioNAICS.csv"
RESULTS_DIR     = "results_ood"
MAX_LENGTH      = 224
QUERY_BATCH     = 8
MAX_EXAMPLES    = 50
USE_FP16        = True
SEED            = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Model:        {MODEL_NAME}  (weights: {MODEL_PATH})")
print(f"OOD dataset:  {OOD_PATH}")
print(f"Taxonomy:     {TAXONOMY_PATH}")
print(f"Max length:   {MAX_LENGTH}")
print(f"Query batch:  {QUERY_BATCH}")
print(f"FP16 weights: {USE_FP16}")
if not os.path.exists(MODEL_PATH):
    print(f"\nWARNING: MODEL_PATH does not exist: {MODEL_PATH}")
    print("Either upload best_model.pt to the Colab session or mount Drive with the trained checkpoint.")

Model:        microsoft/deberta-v3-large  (weights: /content/drive/MyDrive/naics2_large_run3_checkpoints_seed42/best_model.pt)
OOD dataset:  ood_dataset_official.csv
Taxonomy:     ExioNAICS.csv
Max length:   224
Query batch:  8
FP16 weights: True


In [5]:
def clean_text_keep_case(text):
    if pd.isna(text):
        return ""
    s = str(text)
    s = re.sub(r'http\S+|www\.\S+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


df_raw = pd.read_csv(TAXONOMY_PATH)

SECTOR_MERGE = {
    '31': '31-33', '32': '31-33', '33': '31-33',
    '44': '44-45', '45': '44-45',
    '48': '48-49', '49': '48-49',
}

naics2_raw = df_raw[['NAICS_2 Code', 'NAICS_2 Title', 'NAICS_2 Description']].drop_duplicates(subset='NAICS_2 Code').copy()
naics2_raw['NAICS_2 Code'] = naics2_raw['NAICS_2 Code'].astype(str)
naics2_raw['sector_code'] = naics2_raw['NAICS_2 Code'].map(SECTOR_MERGE).fillna(naics2_raw['NAICS_2 Code'])

naics2_corpus = naics2_raw.drop_duplicates(subset='sector_code').copy()
naics2_corpus = naics2_corpus.sort_values('sector_code').reset_index(drop=True)

sector_codes  = naics2_corpus['sector_code'].tolist()
sector_titles = naics2_corpus['NAICS_2 Title'].tolist()
code_to_idx   = {code: i for i, code in enumerate(sector_codes)}
NUM_CLASSES   = len(sector_codes)

naics6_titles = df_raw[['NAICS Code', 'NAICS Title']].drop_duplicates(subset='NAICS Code').dropna()
naics6_titles['NAICS Code'] = naics6_titles['NAICS Code'].astype(str)
naics6_titles['sector'] = naics6_titles['NAICS Code'].str[:2].map(SECTOR_MERGE).fillna(naics6_titles['NAICS Code'].str[:2])

sector_to_examples = {}
for sector, grp in naics6_titles.groupby('sector'):
    titles = [t.strip() for t in grp['NAICS Title'].astype(str).tolist() if t.strip()]
    sector_to_examples[sector] = titles[:MAX_EXAMPLES]

corpus_texts = []
for i, code in enumerate(sector_codes):
    title = sector_titles[i]
    description = naics2_corpus['NAICS_2 Description'].iloc[i]
    description = "" if not pd.notna(description) else str(description)
    description = clean_text_keep_case(description)
    examples = sector_to_examples.get(code, [])
    if examples:
        text = f"{title}. Examples: {'; '.join(examples)}. {description}"
    else:
        text = f"{title}. {description}"
    corpus_texts.append(text)

print(f"NAICS-2 sectors: {NUM_CLASSES}")
for i, (code, title) in enumerate(zip(sector_codes, sector_titles)):
    print(f"  {i:>2}: {code:>5} -> {title}")

print(f"\nEnriched corpus length (chars): "
      f"min={min(len(t) for t in corpus_texts)}, "
      f"max={max(len(t) for t in corpus_texts)}, "
      f"mean={int(np.mean([len(t) for t in corpus_texts]))}")

del df_raw
gc.collect()

NAICS-2 sectors: 20
   0:    11 -> Agriculture, Forestry, Fishing and Hunting
   1:    21 -> Mining, Quarrying, and Oil and Gas Extraction
   2:    22 -> Utilities
   3:    23 -> Construction
   4: 31-33 -> Manufacturing
   5:    42 -> Wholesale Trade
   6: 44-45 -> Retail Trade
   7: 48-49 -> Transportation and Warehousing
   8:    51 -> Information
   9:    52 -> Finance and Insurance
  10:    53 -> Real Estate and Rental and Leasing
  11:    54 -> Professional, Scientific, and Technical Services
  12:    55 -> Management of Companies and Enterprises
  13:    56 -> Administrative and Support and Waste Management and Remediation Services
  14:    61 -> Educational Services
  15:    62 -> Health Care and Social Assistance
  16:    71 -> Arts, Entertainment, and Recreation
  17:    72 -> Accommodation and Food Services
  18:    81 -> Other Services (except Public Administration)
  19:    92 -> Public Administration

Enriched corpus length (chars): min=1133, max=9157, mean=3750


90

In [6]:
df_ood = pd.read_csv(OOD_PATH)
df_ood['naics2_code'] = df_ood['naics2_code'].astype(str)

before = len(df_ood)
df_ood = df_ood[df_ood['naics2_code'].isin(code_to_idx)].reset_index(drop=True)
dropped_unknown = before - len(df_ood)
if dropped_unknown:
    print(f"Dropped {dropped_unknown} OOD rows with sector codes not in trained label space")

df_ood['naics2_idx'] = df_ood['naics2_code'].map(code_to_idx).astype(int)
df_ood['query_text'] = df_ood['model_input'].apply(clean_text_keep_case)

df_ood = df_ood[df_ood['query_text'].str.len() >= 5].reset_index(drop=True)

queries = df_ood['query_text'].tolist()
labels  = df_ood['naics2_idx'].tolist()

print(f"OOD test samples: {len(df_ood)}")
print(f"\nClass distribution (OOD):")
for code in sector_codes:
    cnt = (df_ood['naics2_code'] == code).sum()
    print(f"  {code:>5}  {sector_titles[code_to_idx[code]]:<60}  {cnt}")

print(f"\nQuery length stats (chars):")
print(df_ood['query_text'].str.len().describe().round(0).to_string())
print(f"\nSample query:\n  {queries[0][:200]}")

OOD test samples: 36380

Class distribution (OOD):
     11  Agriculture, Forestry, Fishing and Hunting                    0
     21  Mining, Quarrying, and Oil and Gas Extraction                 30
     22  Utilities                                                     1736
     23  Construction                                                  0
  31-33  Manufacturing                                                 2166
     42  Wholesale Trade                                               0
  44-45  Retail Trade                                                  2563
  48-49  Transportation and Warehousing                                4275
     51  Information                                                   4827
     52  Finance and Insurance                                         5461
     53  Real Estate and Rental and Leasing                            0
     54  Professional, Scientific, and Technical Services              6043
     55  Management of Companies and Enterprises   

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

from transformers import AutoConfig
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=1)
config.hidden_dropout_prob = 0.1
config.attention_probs_dropout_prob = 0.1

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, config=config, torch_dtype=torch.float32
).to(device)

state = torch.load(MODEL_PATH, map_location=device)
if isinstance(state, dict) and 'model_state_dict' in state:
    state = state['model_state_dict']
elif isinstance(state, dict) and 'state_dict' in state:
    state = state['state_dict']
elif isinstance(state, dict) and 'model' in state and isinstance(state['model'], dict):
    state = state['model']

missing, unexpected = model.load_state_dict(state, strict=False)
print(f"Loaded weights from {MODEL_PATH}")
print(f"  Missing keys:    {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")
if missing:
    print(f"  First missing:   {missing[:5]}")
if unexpected:
    print(f"  First unexpected:{unexpected[:5]}")

if USE_FP16 and device.type == "cuda":
    model = model.half()
    print(f"\nConverted model to fp16 for inference.")

model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print(f"\nTotal params: {sum(p.numel() for p in model.parameters()):,}")
if device.type == "cuda":
    torch.cuda.empty_cache()
    used_gb = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM allocated: {used_gb:.2f} GB")

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight     

Loaded weights from /content/drive/MyDrive/naics2_large_run3_checkpoints_seed42/best_model.pt
  Missing keys:    0
  Unexpected keys: 0

Converted model to fp16 for inference.

Total params: 435,062,785
VRAM allocated: 2.61 GB


In [8]:
@torch.no_grad()
def evaluate_ood(model, tokenizer, queries, labels, corpus_texts,
                 max_length=224, query_batch=8):
    model.eval()
    n = len(queries)
    num_classes = len(corpus_texts)

    all_top5 = np.zeros((n, 5), dtype=np.int64)
    all_logits = np.zeros((n, num_classes), dtype=np.float32)

    for start in tqdm(range(0, n, query_batch), desc="Scoring OOD"):
        batch_queries = queries[start:start + query_batch]
        bs = len(batch_queries)

        pair_q = [q for q in batch_queries for _ in range(num_classes)]
        pair_d = corpus_texts * bs

        enc = tokenizer(
            pair_q, pair_d,
            max_length=max_length, truncation=True,
            padding=True, return_tensors='pt'
        )
        enc = {k: v.to(device, non_blocking=True)
               for k, v in enc.items() if k in ['input_ids', 'attention_mask']}

        logits = model(**enc).logits.squeeze(-1)
        logits = logits.float().reshape(bs, num_classes)
        topk = logits.topk(5, dim=1).indices.cpu().numpy()

        all_top5[start:start + bs] = topk
        all_logits[start:start + bs] = logits.cpu().numpy()

    preds = all_top5[:, 0].tolist()
    labels_arr = np.asarray(labels)

    top1 = float(np.mean(all_top5[:, 0] == labels_arr))
    top3 = float(np.mean([labels_arr[i] in all_top5[i, :3] for i in range(n)]))
    top5 = float(np.mean([labels_arr[i] in all_top5[i, :5] for i in range(n)]))
    macro_f1    = f1_score(labels, preds, average='macro',    zero_division=0)
    weighted_f1 = f1_score(labels, preds, average='weighted', zero_division=0)

    return {
        'top1': top1, 'top3': top3, 'top5': top5,
        'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
        'preds': preds, 'top5_indices': all_top5, 'logits': all_logits,
    }


print("Evaluation function ready.")

Evaluation function ready.


In [9]:
results = evaluate_ood(
    model, tokenizer, queries, labels, corpus_texts,
    max_length=MAX_LENGTH, query_batch=QUERY_BATCH,
)

print("\n" + "=" * 70)
print(f"OOD evaluation: {len(queries)} samples, {NUM_CLASSES} classes")
print("=" * 70)
print(f"  Top-1 accuracy: {results['top1']:.4f}")
print(f"  Top-3 accuracy: {results['top3']:.4f}")
print(f"  Top-5 accuracy: {results['top5']:.4f}")
print(f"  Macro F1:       {results['macro_f1']:.4f}")
print(f"  Weighted F1:    {results['weighted_f1']:.4f}")
print("=" * 70)

Scoring OOD:   0%|          | 0/4548 [00:00<?, ?it/s]


OOD evaluation: 36380 samples, 20 classes
  Top-1 accuracy: 0.6856
  Top-3 accuracy: 0.8316
  Top-5 accuracy: 0.8884
  Macro F1:       0.3364
  Weighted F1:    0.7167


In [10]:
target_names = [f"{sector_codes[i]} ({sector_titles[i][:40]})" for i in range(NUM_CLASSES)]
present_idx = sorted(set(labels))

report_str = classification_report(
    labels, results['preds'],
    labels=present_idx,
    target_names=[target_names[i] for i in present_idx],
    digits=4, zero_division=0,
)
print("Per-class classification report:\n")
print(report_str)

report_dict = classification_report(
    labels, results['preds'],
    labels=present_idx,
    target_names=[target_names[i] for i in present_idx],
    digits=4, zero_division=0, output_dict=True,
)

Per-class classification report:

                                               precision    recall  f1-score   support

21 (Mining, Quarrying, and Oil and Gas Extra)     0.0830    0.7000    0.1484        30
                               22 (Utilities)     0.9711    0.3479    0.5123      1736
                        31-33 (Manufacturing)     0.3918    0.8463    0.5357      2166
                         44-45 (Retail Trade)     0.5449    0.2957    0.3834      2563
       48-49 (Transportation and Warehousing)     0.9812    0.8536    0.9129      4275
                             51 (Information)     0.9659    0.4340    0.5989      4827
                   52 (Finance and Insurance)     0.9880    0.9187    0.9521      5461
54 (Professional, Scientific, and Technical )     0.6842    0.9596    0.7988      6043
56 (Administrative and Support and Waste Man)     0.8163    0.4336    0.5664      5440
       62 (Health Care and Social Assistance)     0.9674    0.7309    0.8327      3735
     71 

In [11]:
predictions_df = df_ood[['company name', 'description', 'naics2_code', 'naics2_title']].copy()
predictions_df['pred_idx']    = results['preds']
predictions_df['pred_code']   = predictions_df['pred_idx'].map(lambda i: sector_codes[i])
predictions_df['pred_title']  = predictions_df['pred_idx'].map(lambda i: sector_titles[i])
predictions_df['correct_top1'] = predictions_df['naics2_code'] == predictions_df['pred_code']

for k in range(5):
    predictions_df[f'top{k+1}_code'] = [sector_codes[results['top5_indices'][i, k]] for i in range(len(df_ood))]

predictions_path = os.path.join(RESULTS_DIR, 'ood_predictions.csv')
predictions_df.to_csv(predictions_path, index=False)
print(f"Per-row predictions saved to: {predictions_path}")

summary = {
    'n_samples':         len(queries),
    'n_classes_in_test': len(present_idx),
    'n_classes_total':   NUM_CLASSES,
    'top1':              results['top1'],
    'top3':              results['top3'],
    'top5':              results['top5'],
    'macro_f1':          results['macro_f1'],
    'weighted_f1':       results['weighted_f1'],
    'per_class':         report_dict,
}
summary_path = os.path.join(RESULTS_DIR, 'ood_metrics.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Summary metrics saved to:    {summary_path}")

with open(os.path.join(RESULTS_DIR, 'classification_report.txt'), 'w') as f:
    f.write(f"OOD evaluation on {len(queries)} samples\n")
    f.write(f"Top-1 {results['top1']:.4f} | Top-3 {results['top3']:.4f} | Top-5 {results['top5']:.4f} | "
            f"Macro F1 {results['macro_f1']:.4f} | Weighted F1 {results['weighted_f1']:.4f}\n\n")
    f.write(report_str)
print(f"Classification report saved to: {os.path.join(RESULTS_DIR, 'classification_report.txt')}")

Per-row predictions saved to: results_ood/ood_predictions.csv
Summary metrics saved to:    results_ood/ood_metrics.json
Classification report saved to: results_ood/classification_report.txt


In [12]:
cm = confusion_matrix(labels, results['preds'], labels=list(range(NUM_CLASSES)))

print("Confusion matrix (rows=true, cols=pred), sectors with at least 1 OOD sample:")
header = "true \\ pred  | " + " ".join(f"{c:>5}" for c in sector_codes) + "  |  total"
print(header)
print("-" * len(header))
for i in range(NUM_CLASSES):
    row_total = cm[i].sum()
    if row_total == 0:
        continue
    row = " ".join(f"{cm[i, j]:>5}" for j in range(NUM_CLASSES))
    print(f"{sector_codes[i]:>11}  | {row}  |  {row_total}")

print("\nMost common confusions (true -> pred, at least 30 cases):")
confusions = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm[i, j] >= 30:
            confusions.append((cm[i, j], sector_codes[i], sector_titles[i],
                               sector_codes[j], sector_titles[j]))
confusions.sort(reverse=True)
for cnt, tc, tt, pc, pt in confusions[:20]:
    print(f"  {cnt:>5}  {tc:>5} {tt[:35]:<35} -> {pc:>5} {pt[:35]}")

Confusion matrix (rows=true, cols=pred), sectors with at least 1 OOD sample:
true \ pred  |    11    21    22    23 31-33    42 44-45 48-49    51    52    53    54    55    56    61    62    71    72    81    92  |  total
------------------------------------------------------------------------------------------------------------------------------------------------
         21  |     0    21     0     0     4     2     3     0     0     0     0     0     0     0     0     0     0     0     0     0  |  30
         22  |    17   107   604   220   348    28   127     5     9    17     6   113     3    16    16     1     0     0    73    26  |  1736
      31-33  |    14    92     0    40  1833    78    67     8     1     1     1    14     0     1     1     2     1     0    10     2  |  2166
      44-45  |    62     1     0     7  1020    73   758     1     7     0     3    30     0    63   261    26     6   114   119    12  |  2563
      48-49  |    11     3     0     5   131    41   115  3

In [13]:
import shutil

zip_base = "ood_results"
zip_path = shutil.make_archive(zip_base, 'zip', RESULTS_DIR)
print(f"Zipped {RESULTS_DIR}/ -> {zip_path}")
print(f"Contents:")
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<35}  {size_kb:>10.1f} KB")

try:
    from google.colab import files
    files.download(zip_path)
    print(f"\nDownload triggered for: {zip_path}")
except ImportError:
    print(f"\nNot running in Colab - results are at: {os.path.abspath(zip_path)}")

Zipped results_ood/ -> /content/ood_results.zip
Contents:
  classification_report.txt                   1.5 KB
  ood_metrics.json                            2.9 KB
  ood_predictions.csv                     11190.0 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Download triggered for: /content/ood_results.zip
